# Generation evaluation from first principles

This lab evaluates **generation only** over fixed evidence and controlled outputs. There is no retrieval, model server, database, NLI, score, baseline, or persistence. A case passes only when every visible deterministic check passes.

The advanced end-to-end evaluator remains available in notebook 04. This smaller layer is a learning and diagnosis boundary, not its replacement yet.

## Checkpoint 1 — Objective: See the anatomy of a case

**Run:** the next cell to build three questions with their fixed evidence and explicit expectations.

Text checks use the existing deterministic normalization in this exact order: Unicode NFKC, case folding, Markdown-character removal, Unicode-punctuation removal, whitespace collapse, and edge trimming. Phrase containment happens only after both the answer and expectation go through those steps.

### What to observe

Each case states what evidence exists, what answer text is required or forbidden, whether abstention is expected, the exact used-source set, and the exact model-call count.

### Conclusion

The evaluator has no hidden judgment. Its entire contract is visible in these values.

In [ ]:
from raglab.evaluation import (
    GenerationCase,
    GenerationEvidence,
    GenerationOutput,
    evaluate_generation,
)
from raglab.evaluation.metrics import normalize

cases = {
    "E41": GenerationCase(
        question="What should an operator do if the isolation banner does not appear?",
        evidence=(GenerationEvidence(
            "aster-manual",
            "If the isolation banner does not appear, stop and record fault E41.",
        ),),
        required_phrases=("record fault E41",),
        forbidden_phrases=("fault raised for low flow is E17",),
        expected_source_ids=("aster-manual",),
        expected_model_calls=2,
    ),
    "password": GenerationCase(
        question="How often must Meridian employees rotate their passwords?",
        evidence=(GenerationEvidence(
            "meridian-handbook",
            "The supplied handbook excerpt does not state a password rotation schedule.",
        ),),
        expected_abstained=True,
        expected_model_calls=1,
    ),
    "labels": GenerationCase(
        question="Which proposed labels must be removed?",
        evidence=(GenerationEvidence(
            "orbit-telemetry",
            "Remove user ID and raw request path from the proposed metric labels.",
        ),),
        required_phrases=("user ID", "raw request path"),
        forbidden_phrases=("region", "session ID"),
        expected_source_ids=("orbit-telemetry",),
        expected_model_calls=2,
    ),
}

normalization_examples = {
    "input": "  Record **fault** `Ｅ４１`!  ",
    "normalized": normalize("  Record **fault** `Ｅ４１`!  "),
}
case_anatomy = cases

## Checkpoint 2 — Objective: Prove a correct output passes all five checks

**Run:** the next cell to evaluate controlled outputs that satisfy every expectation.

### What to observe

Every result contains five checks. Each check exposes its name, expected value, observed value, and `PASS`/`FAIL` status. The overall result is `PASS` only because all five are `PASS`.

### Conclusion

There is no percentage or weighted average to hide a broken contract.

In [ ]:
correct_outputs = {
    "E41": GenerationOutput(
        "The operator must record fault E41.", False, ("aster-manual",), 2
    ),
    "password": GenerationOutput(
        "The evidence does not specify a password rotation schedule.", True, (), 1
    ),
    "labels": GenerationOutput(
        "Remove user ID and raw request path.", False, ("orbit-telemetry",), 2
    ),
}

correct_results = {
    name: evaluate_generation(cases[name], output)
    for name, output in correct_outputs.items()
}
assert {name: result.status for name, result in correct_results.items()} == {
    "E41": "PASS", "password": "PASS", "labels": "PASS"
}
correct_results

## Checkpoint 3 — Objective: Diagnose the three recorded failure shapes

**Run:** the next cell to replay one wrong output per case and list only its failed checks.

### What to observe

E41 both omits the required action and repeats the forbidden low-flow answer. Password invents a schedule, fails to abstain, cites evidence, and uses too many calls. Labels contains both required phrases but also the forbidden `region` and `session ID` phrases.

### Conclusion

A failure points to violated contracts directly. Fix the named behavior; do not tune an aggregate score.

In [ ]:
wrong_outputs = {
    "E41": GenerationOutput(
        "The fault raised for low flow is E17.", False, ("aster-manual",), 2
    ),
    "password": GenerationOutput(
        "Rotate passwords every 90 days.", False, ("meridian-handbook",), 2
    ),
    "labels": GenerationOutput(
        "Remove region, user ID, session ID, and raw request path.",
        False,
        ("orbit-telemetry",),
        2,
    ),
}

wrong_results = {
    name: evaluate_generation(cases[name], output)
    for name, output in wrong_outputs.items()
}
failed_checks = {
    name: [check.name for check in result.checks if not check.passed]
    for name, result in wrong_results.items()
}
assert failed_checks == {
    "E41": ["required_phrases", "forbidden_phrases"],
    "password": ["abstained", "used_source_ids", "model_calls"],
    "labels": ["forbidden_phrases"],
}
failed_checks

## Optional appendix — Adapt a future production response

The next rung can run qwen over the **same fixed evidence** and reuse these checks. `GenerationOutput.from_generation_response` reads only the answer, abstention flag, used source IDs, and model-call count. It does not execute retrieval or generation.

The adapter below is defined but deliberately not called: this notebook remains fully hermetic.

In [ ]:
from raglab.generation import GenerationResponse


def adapt_production_response(response: GenerationResponse) -> GenerationOutput:
    return GenerationOutput.from_generation_response(response)
